# Build an Intelligent Warehouse Assistant: Amazon Bedrock Models Through SAP GenAI Hub with LiteLLM and Strands Agents SDK

Welcome! In this tutorial you will learn to build an AI powered warehouse operations assistant combining Amazon Bedrock models (e.g., Amazon Nova Family, Anthropic's Claude models) with SAP GenAI Hub through LiteLLM and the Strands Agents SDK. We will demonstrate how to create agents that query inventory levels and suggest reorder actions; scenarios familiar to S/4HANA and SAP EWM users. You will see how to dynamically switch between Nova Lite for high volume queries and Claude for complex reasoning, all through SAP's centralized model governance. 

## Use Case Scenario 

A manufacturing business which needs instant, intelligent responses to complex inventory questions like:
- "Do we have enough units of product A for the urgent ExampleCorp order?"
- "Which storage bins have the oldest inventory?"
- "Can we fulfill a 200-unit order across all product types?"
- "What's our current capacity utilization?"

In this notebook we will be building an AI agent that connects to SAP APIs and provides human-like responses to complex supply chain queries.

## Prerequisites

1. SAP AI Core credentials in your .env file
2. Strands Agents SDK installed
3. LiteLLM at least version 1.80.10 installed 
4. Your actual SAP_S4HANA_PUBLIC_CLOUD_KEY

## Architecture 


<div style="text-align:center">
    <img src="assets/litellm-warehouse-agent.jpeg" width="65%" />
</div>

## 1. Load Credentials and Invoke Models Through SAP Generative AI Hub

We will be using the open-source framework [Strands Agents SDK](https://strandsagents.com/latest/) to build our agent. As Large Language Model we will be using a variety of models including models from the Claude and Nova model family. We will be using [LiteLLM](https://docs.litellm.ai/) to invoke models deployed in SAP's [Generative AI Hub](https://www.sap.com/products/artificial-intelligence/generative-ai-hub.html).

To invoke models through Generative AI Hub, you will need to have AI Core credentials. To get them, create a service key from [here](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/create-service-key?locale=en-US). 

This is how it should look according to the [LiteLLM documentation](https://sap-contributions.github.io/litellm-agentic-examples/_notebooks/examples/aws_strands.html#Credentials-for-SAP-Gen-AI-Hub): 

```
AICORE_AUTH_URL="https://* * * .authentication.sap.hana.ondemand.com/oauth/token"
AICORE_CLIENT_ID=" *** "
AICORE_CLIENT_SECRET=" *** "
AICORE_RESOURCE_GROUP=" *** "
AICORE_BASE_URL="https://api.ai.***.cfapps.sap.hana.ondemand.com/
```

The credentials should then be savind in a .env file in this directory. 

In [ ]:
#!pip install 'litellm>=1.83.0' 'strands-agents[litellm]'

In [ ]:
#imports

#from dotenv import load_dotenv 
from strands import Agent, tool 
from strands.models.litellm import LiteLLMModel

In [ ]:
# loading credentials from .env file
#load_dotenv()

Let's try out Strands agents with an easy example invoking different models in SAP's Generative AI Hub:

In [ ]:
# create agent using the Strands Agents SDK 

agent = Agent(
    system_prompt="You are a helpful assistant",
    model=LiteLLMModel(
        model_id="sap/amazon--nova-micro")
)
response = agent("Tell me about agentic AI")
print(response)

In [ ]:
# let's use a different model, you can find a list of all models available here: https://me.sap.com/notes/3437766
# let's also specify some parameters of the model 

agent = Agent(
    system_prompt="You are a helpful assistant",
    model=LiteLLMModel(model_id="sap/anthropic--claude-4.5-sonnet",
        params={
        "max_tokens": 1000,
        "temperature": 0.7, 
        }
    )
)
response = agent("Tell me about agentic AI")
print(response)

## 2. Build an Intelligent Warehouse Assistant




## 2a. Create Selector Sub-Agent and Configure as Tool

The Selector Sub-Agent acts as an intelligent API router that analyzes user queries and automatically determines which SAP OData API and endpoint should be used to fulfill the request. By loading and parsing OpenAPI specification files from the knowledgebase directory, it builds a comprehensive understanding of available APIs, their capabilities, endpoints, and base URLs. When given a user query, the selector agent evaluates the query against all available API specifications and provides a reasoned recommendation for which API and specific endpoint is most appropriate, including the correct sandbox base URL to use. This intelligent routing capability enables the main warehouse agent to dynamically work with multiple SAP APIs without hardcoded API selections, making the system more flexible and maintainable as new APIs are added to the knowledgebase.

In [ ]:
# import statements
from pathlib import Path
import yaml

In [ ]:

# Load YAML OpenAPI files from directory
def load_openapi_specs(path="./assets/knowledgebase"):
    specs = []
    for file in Path(path).glob("*.y*ml"):
        with open(file, "r") as f:
            try:
                data = yaml.safe_load(f)
                servers = data.get("servers", [])
                base_urls = []
                for s in servers:
                    url = s.get("url")
                    desc = s.get("description", "")
                    if url:
                        base_urls.append({"url": url, "description": desc})
                summary = {
                    "file": file.name,
                    "title": data.get("info", {}).get("title"),
                    "description": data.get("info", {}).get("description"),
                    "paths": list(data.get("paths", {}).keys()),
                    "base_urls": base_urls or [{"url": "/", "description": "Default"}]
                }
                specs.append(summary)
            except Exception as e:
                print(f"Error parsing {file}: {e}") 
    return specs

def specs_to_prompt_string(specs):
    prompt_parts = []
    for spec in specs:
        base_url_str = ", ".join(
            f"{url_info['url']} ({url_info['description']})" for url_info in spec['base_urls']
        )
        paths_str = ", ".join(spec['paths'])
        part = (
            f"API Spec: {spec['file']}\n"
            f"Title: {spec['title']}\n"
            f"Description: {spec['description']}\n"
            f"Base URLs: {base_url_str}\n"
            f"Endpoints: {paths_str}\n"
        )
        prompt_parts.append(part)
    return "\n---\n".join(prompt_parts)

# Usage example:
specs = load_openapi_specs()
specs_prompt_string = specs_to_prompt_string(specs)

SELECTOR_SYSTEM_PROMPT = f"""
You are an API selection subagent.
Given a user query and the following list of OpenAPI specs, each with a title, description, base URLs, and available endpoints:

{specs_prompt_string}

Your task is to identify which API and specific endpoint is most appropriate to fulfill the user query.
Provide clear reasoning for your choice, referencing the API spec details provided.
If no suitable API is found, explain why.

Make sure to reply with the sandbox BASE URL.

"""

# Strands subagent for selection
selector_agent = Agent(
    system_prompt=SELECTOR_SYSTEM_PROMPT,
    model=LiteLLMModel(
        model_id="sap/amazon--nova-micro")
)

# query = "Get all inventory stock for item WM-AN02 "
query = "Get freight information for item WM-AN02 "

response = selector_agent(query)

print(response)


In [ ]:
# Configure the selector sub agent as tool for our main warehouse agent 

@tool
def SelectorAPIAgentAsATool(query: str) -> str:
    """
    This analyzes available OpenAPI specs and selects the most appropriate
    API and endpoint based on user queries.

    Args:
        query: User query describing what they want to accomplish

    Returns:
        Agent response with API selection recommendation if successful, 
        error message if initialization fails
    """
    return selector_agent(query).message

## 2b. Import the Odata_tool from the Util Folder

The **odata_caller** (Universal OData Tool) runs OData operations (GET, POST, PUT, DELETE) with automatic authentication. Supports OData query parameters ($filter, $select, $orderby, $top) and structured error handling.


In [ ]:
# import odata tool from utils folder
from util.odata_tool import odata_caller

#other import statements
import os 
import getpass

## 2c. Create Warehouse Agent

We will now create the intelligent warehouse operations agent with dynamic ODAta exploration capabilities. Also, you need your SAP S4HANA PUBLIC CLOUD KEY handy in the next step. You can create and copy an S/4 HANA Public Cloud Test API key by selecting Register here or logging in with a free SAP account [here](https://api.sap.com/api/CE_WHSEPHYSICALSTOCKPRODUCTS_0001/tryout). Note, you may need to click the above link again if you have recently created a new free account to get the API Key.

In [ ]:
# Prompt the user to securely input the API key
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")

In [ ]:
# Define the enhanced system prompt for our warehouse agent
WAREHOUSE_SYSTEM_PROMPT = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750). 
You have access to real-time SAP warehouse data through dynamic OData API exploration capabilities.

CORE CAPABILITIES:
1. API Structure Exploration: Dynamically discover available data fields and entities
2. Product Discovery: Find available products without hardcoded assumptions
3. Dynamic Querying: Construct intelligent OData queries based on user needs

APPROACH TO PROBLEM SOLVING:
- You must start by using the SelectorAPIAgentAsATool to help you determine which OData API to call
- Next use the $metadata endpoint that to understand the API that SelectorAPIAgentAsATool provides  
- Use dynamic queries to discover information rather than making assumptions
- Leverage OData filtering, sorting, and selection to get precise answers
- You may need to do this multiple times. Always write what URL have constructed so user is informed.

PRODUCT KNOWLEDGE (can be expanded through discovery):
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Explain your discovery process when exploring new data
- Provide specific, actionable insights with quantitative data
- Do not use emojis

When users ask questions:
1. First determine what data you need to answer the question
3. Feel free to use the odata_caller tool as many times as needed to get the right information.
3. Construct appropriate OData queries to get the specific information needed
4. Analyze the results and provide comprehensive, intelligent responses

Example for using the odata_caller tool:
```python
        odata_caller(
            base_url="",
            endpoint="WarehouseStockProducts",
            operation="get",
            odata_params={"$filter": "Product eq 'WM-AN02'"},
            auth_type="api_key",
            auth_env_var="SAP_S4HANA_PUBLIC_CLOUD_KEY"
        )
        ```

Focus on SAP S/4 HANA OData endpoints, warehouse management APIs, and supply chain operations.

Available tools:
- odata_caller: Universal OData tool for SAP API interactions with built-in authentication and query parameter support
- auth_token: Use os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY") to get the API key
Example below:

# For SAP OData calls, use consistent headers
headers = {
    "APIKey": os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY"),
    "Accept": "application/json", OR "application/xml" choose as needed 
    "DataServiceVersion": "2.0"
}

The odata_caller tool handles SAP-specific authentication automatically and provides comprehensive error handling and response formatting.

"""
        


# Create the enhanced warehouse operations agent
warehouse_agent = Agent(
    model=LiteLLMModel(model_id="sap/anthropic--claude-4.5-sonnet",
        params={
        "max_tokens": 1000,
        "temperature": 0.7, 
        }
    ),
    tools=[
        # ODataExecutorAgentAsATool,
        SelectorAPIAgentAsATool,
        odata_caller, # For OData REST API calls
    ],
    system_prompt=WAREHOUSE_SYSTEM_PROMPT
)

In [ ]:
# Quick test to verify the agent is working
try:
    test_response = warehouse_agent("Hello, can you introduce yourself and your capabilities?")
    response_text = str(test_response)

    print(f"\n Agent Test: {response_text[:150]}...")
except Exception as e:
    print(f"\n️ Agent test failed: {e}")
    print("Agent created successfully, but test response had an issue.")

# 3. Try Out Your Intelligent Warehouse Agent

Let's test our enhanced agent with realistic warehouse management scenarios. The agent will now dynamically explore the API to answer questions.

## 3a. Scenario 1: Complete Inventory Overview

In [ ]:
# Test inventory overview with dynamic discovery
response = warehouse_agent(
    "I need a complete overview of our current warehouse inventory. "
    "What products do we have, how much of each, and what's our overall stock situation?"
)

## 3b. Scenario 2: Urgent Order Fulfillment Check

In [ ]:
# Test urgent order scenario with dynamic discovery
warehouse_agent.messages = []
response = warehouse_agent(
    "We just received an urgent order from ExampleCorp for 200 units of WM-AN02 Control Units. "
    "Can we fulfill this order immediately? If yes, which storage bins should our picking team target first?"
)